In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy as sp
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
selected_cols = [
    'Player', 'Pos','Squad','Comp','MP','Min',
    'Gls','Ast','G+A','xG','xAG', 'npxG' ,
    'Sh','SoT','SoT%','KP', 'Succ%',
    'SCA','GCA'
]
df= pd.read_csv(r"D:\Salah_Performance\Dataset\players_data_light-2024_2025.csv" , usecols = selected_cols)

In [3]:
df.sample(5)

,Player,Pos,Squad,Comp,MP,Min,Gls,Ast,G+A,xG,npxG,xAG,Sh,SoT,SoT%,KP,SCA,GCA,Succ%
845,Endrick,FW,Real Madrid,es La Liga,22,363,1,0,1,2.5,2.5,0.5,24,6,25.0,4,8,2,36.4
1645,Arnau Martinez,"DF,MF",Girona,es La Liga,32,2624,2,2,4,1.2,1.2,2.2,19,6,31.6,18,56,6,54.1
2611,David Torres,DF,Valladolid,es La Liga,18,1387,0,0,0,0.3,0.3,0.0,3,1,33.3,1,3,0,50.0
521,Franco Cervi,"FW,MF",Celta Vigo,es La Liga,7,273,0,0,0,0.0,0.0,0.3,0,0,NaN,5,5,0,50.0
1941,Wilson Odobert,"FW,MF",Tottenham,eng Premier League,16,850,1,0,1,1.6,1.6,1.0,13,4,30.8,6,16,1,37.5


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2854 entries, 0 to 2853
Data columns (total 19 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Player  2854 non-null   object 
 1   Pos     2854 non-null   object 
 2   Squad   2854 non-null   object 
 3   Comp    2854 non-null   object 
 4   MP      2854 non-null   int64  
 5   Min     2854 non-null   int64  
 6   Gls     2854 non-null   int64  
 7   Ast     2854 non-null   int64  
 8   G+A     2854 non-null   int64  
 9   xG      2854 non-null   float64
 10  npxG    2854 non-null   float64
 11  xAG     2854 non-null   float64
 12  Sh      2854 non-null   int64  
 13  SoT     2854 non-null   int64  
 14  SoT%    2335 non-null   float64
 15  KP      2854 non-null   int64  
 16  SCA     2854 non-null   int64  
 17  GCA     2854 non-null   int64  
 18  Succ%   2411 non-null   float64
dtypes: float64(5), int64(10), object(4)
memory usage: 423.8+ KB


In [5]:
df.isna().sum()

Player      0
Pos         0
Squad       0
Comp        0
MP          0
Min         0
Gls         0
Ast         0
G+A         0
xG          0
npxG        0
xAG         0
Sh          0
SoT         0
SoT%      519
KP          0
SCA         0
GCA         0
Succ%     443
dtype: int64

In [6]:
len(df)

2854

In [7]:
df.shape

(2854, 19)

<h1> Cleaning </h1>

In [8]:
df.rename(columns={
    'G+A': 'GA',
    'SoT%': 'SoT_pct',
    'Succ%': 'Succ_pct'
}, inplace=True)

In [9]:
df['SoT_pct'] = df['SoT_pct'].fillna(0)
df['Succ_pct'] = df['Succ_pct'].fillna(0)

In [10]:
df.drop_duplicates(inplace=True)

In [11]:
df = df[df['Min'] >= 300]

In [12]:
df['Goals_per90'] = round(df['Gls'] / df['Min'] * 90 , 2)
df['Assists_per90'] = round(df['Ast'] / df['Min'] * 90 , 2)
df['XG_per90'] = round(df['xG'] / df['Min'] * 90 , 2)


In [13]:
df['Finishing'] = round(df['Gls'] - df['xG'] , 2)

In [14]:
df['Shot_Accuracy'] = round((df['SoT'] / df['Sh']) * 100, 2)

In [15]:
df.replace([np.inf , -np.inf] , 0 , inplace=True)

<h1> Analysis </h1>

In [16]:
salah = df[df['Player'] == 'Mohamed Salah']
salah

,Player,Pos,Squad,Comp,MP,Min,Gls,Ast,GA,xG,...,SoT_pct,KP,SCA,GCA,Succ_pct,Goals_per90,Assists_per90,XG_per90,Finishing,Shot_Accuracy
2304,Mohamed Salah,FW,Liverpool,eng Premier League,38,3371,29,18,47,25.2,...,41.3,88,169,27,42.3,0.77,0.48,0.67,3.8,41.32


In [17]:
salah_goals = salah['Gls'].values[0]
salah_goals

29

In [18]:
salah_assists = salah['Ast'].values[0]
salah_assists

18

In [19]:
salah_matches = salah['MP'].values[0]
salah_matches

38

In [21]:
on_target = salah['SoT'].values[0]
off_target = (salah['Sh'].values[0] - salah['SoT'].values[0])
print('ON Target = ' , on_target)
print('OFF Target = ' , off_target)


ON Target =  50
OFF Target =  71


In [23]:
goals = salah['Gls'].values[0]
xg = salah['xG'].values[0]
print(goals ,',' , xg)

29 , 25.2


Salah >> Overperforming , Good finishing

In [24]:
Assists = salah['Ast'].values[0]
Chances = salah['GCA'].values[0]
Key_pass = salah['KP'].values[0]
print(Assists ,',' , Chances , ',' , Key_pass)

18 , 27 , 88


Liverpool attackers underperform the quality of chances created by Salah.

In [29]:
Top_soccers = df[['Player' , 'Gls']].sort_values(by='Gls' , ascending=False).head(10)
Top_soccers

,Player,Gls
1691,Kylian Mbappé,31
2304,Mohamed Salah,29
1483,Robert Lewandowski,27
1317,Harry Kane,26
2201,Mateo Retegui,25
1219,Alexander Isak,23
1109,Erling Haaland,22
405,Ante Budimir,21
697,Ousmane Dembélé,21
1060,Mason Greenwood,21


- Comparing Salah to the best wingers (contributions)
- Comparing Salah to the best playmakers (key passes)

In [30]:
df.to_csv(r'D:\Salah_Performance\Data_After_Transform.csv', index=False)